In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline 

In [ ]:
df=pd.read_csv("../input/fortnite-statistics-corresponding-to-mental-state/FortniteStatistics.csv",engine="python");df.head()

In [ ]:
df["Time of Day"].unique()

In [ ]:
df.shape

In [ ]:
df1=df.describe().transpose();df1

In [ ]:
df.info()

In [ ]:
df.Accuray=df.Accuracy.replace(regex=True, inplace=True, to_replace=r'[^0-9.\-]', value=r'')
df.Accuracy=df.Accuracy.astype("float")
df.dtypes

In [ ]:
df.describe().boxplot()
plt.xticks(rotation=90)
plt.show()

In [ ]:
f,(ax1,ax2) = plt.subplots(1,2,sharey=True,figsize=(25,10))
g1 = sns.heatmap(df[df["Mental State"]=="sober"].corr()*100,cmap="YlGnBu",cbar=False,ax=ax1,annot=True)
g1.set_title('features when mental state was sober')
g2 = sns.heatmap(df[df["Mental State"]=="high"].corr()*100,cmap="YlGnBu",cbar=True,ax=ax2,annot=True)
g2.set_title('features when mental state was high')
plt.suptitle("Heatmap as per their Mental state ")
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()
df["Mental State"]=labelencoder.fit_transform(df['Mental State']).astype("int")
df["Accuracy"]=labelencoder.fit_transform(df['Accuracy']).astype("float")
df["Time of Day"]= [i.replace(':00 PM', '') for i in df["Time of Day"]]
df["Time of Day"]=pd.to_numeric(df["Time of Day"], errors='ignore')

In [ ]:
df.dtypes

In [ ]:
df.iloc[:,4:].apply(lambda x: (x-x.mean())/ x.std()).plot.box()#Normalize the columns to see the unbiaseness in short perspective
plt.xticks(rotation=90) # player is an average one.
#most of time he played in his zone vary less outliers

In [ ]:
sns.pairplot(df, hue="Mental State",diag_kind="hist",corner=True);plt.show()

* ## Looks like our Player become high,active in/btw 4 to 9
* ## Players Performance decreased as the day comes to end.
* ## He was at his peak at the start of the game.
* ## Accuracy,revives,assists goes down when the elminations were high.

### Above given pairplot shows clear picture of data.

In [ ]:
sns.catplot(x="Time of Day",col="Mental State",y="Placed",data=df)# 0 means "high" and 1 means "Sober"

# Factors that decides his placed or Position in Fortnite points table are
1. 'Time of Day', 'Eliminations','Assists', 'Revives', 'Accuracy', 'Hits', 'Head Shots','Distance Traveled', 'Materials Gathered', 'Materials Used','Damage Taken', 'Damage to Players', 'Damage to Structures'

In [ ]:
x=df.loc[:,["Placed",'Time of Day', 'Eliminations','Assists', 'Revives','Accuracy',
            'Hits', 'Head Shots','Distance Traveled', 'Materials Gathered',
            'Materials Used','Damage Taken', 'Damage to Players', 'Damage to Structures']]
y=df["Mental State"]

In [ ]:
from sklearn.feature_selection import mutual_info_classif
important_features =mutual_info_classif(x,y)
feat_importances=pd.Series(important_features,df.columns[0:len(df.columns)-2])
feat_importances.plot(kind="bar")

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(random_state=0,n_estimators=100)
clf.fit(x_train, y_train)
y_pred=clf.predict(x_test);y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(accuracy_score(y_test, y_pred)*100)

In [ ]:
#Lets try Neural Network regressor
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, Activation,Dropout
from keras.models import Sequential

In [ ]:
X=df.loc[:,["Mental State",'Time of Day', 'Eliminations','Assists', 'Revives','Accuracy',
            'Hits', 'Head Shots','Distance Traveled', 'Materials Gathered',
            'Materials Used','Damage Taken', 'Damage to Players', 'Damage to Structures']]
Y=df.Placed

In [ ]:
# Splitting the dataset into the Training set and Test set
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.25, random_state = 0)

In [ ]:
plt.figure(num=None, figsize=(10,8), dpi=80, facecolor='whitesmoke', edgecolor='pink')

feat_importances = pd.Series(clf.feature_importances_, index= X.columns)

feat_importances.nlargest(10).plot(kind='barh')

In [ ]:
input_size=14
output_size=1
models = tf.keras.Sequential()
models.add(Dense(10, activation = 'relu', input_dim = input_size))
models.add(Dense(units = 5, activation = 'sigmoid'))
models.add(Dropout(0.2))
    # Adding the third hidden layer
models.add(Dense(units = 5, activation = 'relu'))
models.add(Dropout(0.2))

models.add(Dense(units = 5, activation = 'relu'))
models.add(Dense(units = 5, activation = 'relu'))
    # Adding the output layer
models.add(Dense(units = 1))
models.compile(optimizer = tf.keras.optimizers.SGD(learning_rate=0.02), loss = 'mean_squared_error')
history=models.fit(X,Y,epochs=200,verbose=1)
history

In [ ]:
loss=history.history['loss']
epochs = range(1, len(loss)+1)
plt.plot(epochs, loss, 'go', label='Training loss')
plt.legend()
plt.show()

In [ ]:
#model didn't learn well

In [ ]:
#lets try random forest regressor

from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(n_estimators = 100, random_state = 0)
  
# fit the regressor with x and y data
regressor.fit(X_train, Y_train)  
Y_pred = regressor.predict(X_test)

In [ ]:
df_NEW=pd.DataFrame({'Actual':Y_test, 'Predicted':Y_pred})

In [ ]:
df_NEW

In [ ]:
from sklearn import metrics
print('Mean Absolute Error:', metrics.mean_absolute_error(Y_test, Y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(Y_test, Y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(Y_test, Y_pred)))

In [ ]:
errors = abs(Y_pred - Y_test)
print('Mean Absolute Error:', round(np.mean(errors), 2), 'degrees.')

mape = 100 * (errors / Y_test)
accuracy = 100 - np.mean(mape)
print('Accuracy:', round(accuracy, 2), '%.')

In [ ]:
plt.figure(figsize=(5, 7))


ax = sns.distplot(Y_test, hist=False, color="r", label="Actual Value")
sns.distplot(Y_pred, hist=False, color="b", label="Fitted Values" , ax=ax)


plt.title('Actual vs Fitted Values')


plt.show()

In [ ]:
# IMPROVE IT LATER